## Interactive Showcase: Physics-Aware 3D Inverse Design
**Test-Time Optimization (TTO) Pipeline**

This notebook provides an interactive demonstration of the `ConditionalSurfaceDecoder` architecture. It showcases how abstract, multimodal physical constraints are projected into a learned latent space to guide the generation of continuous 3D Signed Distance Fields (SDFs).

> **Note:** This demonstration utilizes a lightweight, synthetic pre-trained checkpoint (`dummy_weights.pth`) designed purely for architectural validation. The underlying geometric representation is a simplified sphere, allowing for rapid execution on CPU/GPU without exposing proprietary high-fidelity training data.


In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Ensure the root project directory is in the path to import our modules
sys.path.append(os.path.abspath('..'))

from inertiasdf.model_architecture import ConditionalSurfaceDecoder
from inertiasdf.inference_api import LatentOptimizationWrapper
from inertiasdf.geometry_utils import extract_marching_cubes

# Hardware configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing pipeline on: {device}")


## 1. Network Initialization & Weight Loading
We initialize the obfuscated network topology and load the synthetic baseline weights. The network is then wrapped in our `LatentOptimizationWrapper`, which freezes the network parameters and isolates the stochastic latent vector $z$ as the sole optimizable variable.


In [ ]:
# 1. Initialize base architecture
base_model = ConditionalSurfaceDecoder(num_shapes=1, latent_dim=64).to(device)

# 2. Load synthetic checkpoint from the weights directory
weights_path = os.path.join("..", "weights", "dummy_weights.pth")
base_model.load_state_dict(torch.load(weights_path, map_location=device))

# 3. Wrap model for Test-Time Optimization (Freezes weights, exposes 'z')
wrapper = LatentOptimizationWrapper(base_model).to(device)
print("Network initialized and frozen. Latent space exposed for TTO.")

## 2. Defining Multimodal Target Constraints
In this step, we define the abstract physical constraints the unseen geometry must satisfy. This includes continuous numeric tensors (e.g., moments of inertia, mass, etc), bounding spatial limits, and categorical identifiers.

In [ ]:
# Setup generalized target conditions for the demonstration
target_conditions = {
    'constraints': torch.randn(1, 10).to(device),          # Continuous numeric conditions
    'bounds': torch.tensor([[0.5, 0.5, 0.2]]).to(device),  # Spatial bounding limits
    'category': torch.tensor([4]).to(device)               # Categorical material/type ID
}

# Generate point cloud samples to compute spatial gradients during optimization
num_samples = 2000
sample_pts = (torch.rand(num_samples, 3, device=device) * 2.0) - 1.0
sample_sdf = (torch.norm(sample_pts, dim=1) - 0.4).view(-1, 1) # Target proxy representation

print("Target constraints generated successfully.")


## 3. Test-Time Optimization (TTO) Loop
Instead of optimizing a discrete 3D mesh (which is computationally expensive and prone to topological errors), we perform gradient descent directly on the low-dimensional latent vector $z$. The neural network projects this optimized latent vector into the physical 3D space.

In [ ]:
def run_tto(epochs=300, lr=0.05):
    print("Starting Latent Optimization...")
    # Initialize latent vector with small random noise
    torch.nn.init.normal_(wrapper.z, mean=0.0, std=0.01)

    # Optimizer targets ONLY the latent vector
    optimizer = torch.optim.Adam([wrapper.z], lr=lr)

    # Expand conditions for parallel point evaluation
    const_exp = target_conditions['constraints'].expand(num_samples, -1)
    bounds_exp = target_conditions['bounds'].expand(num_samples, -1)
    cat_exp = target_conditions['category'].expand(num_samples).long()

    wrapper.train()

    for epoch in range(epochs):
        optimizer.zero_grad()

        # Forward pass: Project constraints and latent space to spatial domain
        pred_sdf = wrapper(sample_pts, const_exp, bounds_exp, cat_exp)

        # Loss: Geometric alignment + Latent Regularization (Curriculum Learning proxy)
        loss = torch.nn.functional.l1_loss(pred_sdf, sample_sdf) + (1e-4 * torch.norm(wrapper.z))

        # Backpropagation to the latent vector z
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 30 == 0:
            print(f"Epoch [{epoch+1:03d}/{epochs}] | Latent Loss: {loss.item():.6f}")

    print("Optimization Complete.")

# Execute the TTO pipeline
run_tto(epochs=150)

## 4. 3D Surface Extraction & Visualization
With the optimal latent representation found, we query the neural network across a dense 3D spatial grid. We then apply the **Marching Cubes** algorithm to extract the continuous zero-level set (the physical surface) for visualization.

In [ ]:
print("Extracting physical mesh via Marching Cubes...")
# Use a moderate resolution for rapid interactive plotting
verts, faces = extract_marching_cubes(wrapper, target_conditions, grid_res=60)

if len(verts) > 0:
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Create 3D polygon collection
    mesh_plot = Poly3DCollection(verts[faces], alpha=0.9)
    mesh_plot.set_edgecolor('k')
    mesh_plot.set_linewidth(0.3)
    mesh_plot.set_facecolor('#1f77b4')

    ax.add_collection3d(mesh_plot)

    # Set tight bounding box for "zoom" effect
    zoom = 0.55
    ax.set_xlim([-zoom, zoom])
    ax.set_ylim([-zoom, zoom])
    ax.set_zlim([-zoom, zoom])

    ax.axis('off')
    plt.title("Optimized Target Geometry", fontsize=16, pad=15)
    plt.show()
else:
    print("Warning: No surface found. Try restarting the optimization with a different seed.")